In [2]:
# ======================================================
# Project: Unified Military Analytics & Comparison Dashboard
# Module 1: Data Collection (Web Scraping)
# Description:
# - Scrapes country names and global rank
# - Scrapes all military & economic metrics
# - Cleans numeric values
# - Outputs a single unified CSV
# ======================================================

import requests
from bs4 import BeautifulSoup
import pandas as pd
import re

# -----------------------------------
# Global request header
# -----------------------------------
HEADERS = {
    "User-Agent": "Mozilla/5.0"
}

# -----------------------------------
# Read metric links from txt file
# -----------------------------------
def read_metric_links(file_path):
    with open(file_path, "r", encoding="utf-8") as f:
        return [line.strip() for line in f if line.startswith("http")]

# -----------------------------------
# Scrape Country Name + Global Rank
# -----------------------------------
def scrape_country_rank():
    url = "https://www.globalfirepower.com/countries-listing.php"
    response = requests.get(url, headers=HEADERS)
    soup = BeautifulSoup(response.text, "html.parser")

    data = []

    for box in soup.select("div.picTrans.recordsetContainer"):
        try:
            country = box.find(
                "span", class_="textWhite textLarge textShadow"
            ).text.strip()

            rank = box.find(
                "span", class_="textWhite textLarge textBold"
            ).text.strip()

            data.append([country, rank])
        except:
            continue

    return pd.DataFrame(data, columns=["Country", "Rank"])

# -----------------------------------
# Scrape all metric pages
# -----------------------------------
def scrape_metrics(base_df, metric_links):

    for url in metric_links:
        print("Scraping:", url)

        response = requests.get(url, headers=HEADERS)
        soup = BeautifulSoup(response.text, "html.parser")

        rows = []

        for box in soup.select("div.picTrans.recordsetContainer"):
            try:
                country = box.find(
                    "span", class_="textWhite textLarge textShadow"
                ).text.strip()

                value = box.find_all(
                    "span", class_="textWhite textLarge"
                )[-1].text.strip()

                rows.append([country, value])
            except:
                continue

        column_name = url.split("/")[-1].replace(".php", "")

        temp_df = pd.DataFrame(rows, columns=["Country", column_name])

        # Merge metric into main dataframe
        base_df = base_df.merge(temp_df, on="Country", how="left")

    return base_df

# -----------------------------------
# Clean numeric columns
# -----------------------------------
def clean_numeric_values(df):

    for col in df.columns:
        if col in ["Country", "Rank"]:
            continue

        df[col] = (
            df[col]
            .astype(str)
            .str.replace(",", "", regex=False)      # remove commas
            .str.replace(r"\s+", "", regex=True)    # remove tabs/newlines
            .str.extract(r"(-?\d+\.?\d*)")[0]       # extract numbers only
        )

    return df

# -----------------------------------
# RUN PIPELINE
# -----------------------------------
metric_links = read_metric_links("links_for_global_military_data.txt")

df = scrape_country_rank()
df = scrape_metrics(df, metric_links)
df = clean_numeric_values(df)

# Save final output
df.to_csv("global_military_data_final.csv", index=False)

print("\n✅ SCRAPING COMPLETED SUCCESSFULLY")
print("Total Countries:", len(df))
print("Total Columns:", len(df.columns))
print(df.head())


Scraping: https://www.globalfirepower.com/total-population-by-country.php
Scraping: https://www.globalfirepower.com/available-military-manpower.php
Scraping: https://www.globalfirepower.com/manpower-fit-for-military-service.php
Scraping: https://www.globalfirepower.com/manpower-reaching-military-age-annually.php
Scraping: https://www.globalfirepower.com/active-military-manpower.php
Scraping: https://www.globalfirepower.com/active-reserve-military-manpower.php
Scraping: https://www.globalfirepower.com/manpower-paramilitary.php
Scraping: https://www.globalfirepower.com/capital-cities-by-total-population.php
Scraping: https://www.globalfirepower.com/aircraft-total.php
Scraping: https://www.globalfirepower.com/aircraft-total-fighters.php
Scraping: https://www.globalfirepower.com/aircraft-total-attack-types.php
Scraping: https://www.globalfirepower.com/aircraft-total-transports.php
Scraping: https://www.globalfirepower.com/aircraft-total-trainers.php
Scraping: https://www.globalfirepower.co